# GRPO Training (LABD-2.1)

Loads the pre-built recovery dataset from Notebook 1 and runs GRPO with three reward functions.

**Reward design (locked):**
| Reward | Signal | Value |
|---|---|---|
| Correctness | `<execute>` code passes all tests | **+3** else 0 |
| Stuck-think | `<think>` opened, `</think>` missing | **−1** else 0 |
| Format | Hallucinated `<tool_response>` → **−2**; no `<execute>` and not stuck-think → **−1**; else 0 |

Correctness dominates. Penalties are small, non-overlapping, and target the three observed SFT failure modes (markdown drift, hallucinated feedback, stuck-think).

**Generation:** Qwen3 sampling at **T=0.7** (slightly more diverse than data-gen T=0.6 to give GRPO useful intra-group spread). No NoWait at training time — the stuck-think reward needs the failure mode to occur in order to teach against it.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["UNSLOTH_VLLM_STANDBY"]    = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_USE_AOT_COMPILE"]    = "0"

# Override cpu_count for Modal allocations
import multiprocessing
multiprocessing.cpu_count = lambda: 4

In [ ]:
%%capture
import os, glob, shutil

# Modal: fix missing curand headers
for h in glob.glob("/usr/local/lib/python3.12/site-packages/nvidia/curand/include/*.h"):
    dest = f"/usr/local/cuda/include/{os.path.basename(h)}"
    if not os.path.exists(dest):
        os.symlink(h, dest)

# Clear stale flashinfer build cache
for d in [os.path.expanduser("~/.cache/flashinfer"),
          "/root/unsloth_compiled_cache/.cache/flashinfer"]:
    if os.path.exists(d):
        shutil.rmtree(d)

os.environ["UNSLOTH_VLLM_STANDBY"]    = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install --upgrade pip
!pip install --upgrade "peft<=0.18.2"
!pip install unsloth vllm -qqq
!pip install langid -qq
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!pip install pydantic==2.11.5 pydantic_core==2.33.2 --force-reinstall --no-deps
!pip install --upgrade --force-reinstall --no-deps "protobuf>=6.33.5"

In [ ]:
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["VLLM_USE_AOT_COMPILE"] = "0"

import io
import sys
import re
import json
import gc
import signal
import random
import subprocess
import tempfile
import concurrent.futures

import torch
torch.multiprocessing.set_start_method("spawn", force=True)

from unsloth import FastLanguageModel
from datasets import load_from_disk, Dataset
from trl import GRPOConfig, GRPOTrainer

## 1. Model loading (vLLM enabled for GRPO generations)

In [ ]:
import os
import huggingface_hub

hf_token = os.environ["HF_TOKEN"]
huggingface_hub.login(token=hf_token)

In [ ]:
MODEL_NAME    = "moazeldegwy/Qwen3-0.6B-LABD-2.1-SFT"
MAX_SEQ_LEN   = 6144
LORA_RANK     = 16

MAX_PROMPT_LEN     = 3072
MAX_COMPLETION_LEN = 3072

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    max_seq_length         = MAX_SEQ_LEN,
    dtype                  = torch.bfloat16,
    load_in_4bit           = False,
    fast_inference         = True,
    max_lora_rank          = LORA_RANK,
    gpu_memory_utilization = 0.85,
    enforce_eager          = True,
)

tokenizer.padding_side    = "right"
tokenizer.truncation_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = LORA_RANK * 2,
    use_gradient_checkpointing = "unsloth",
    random_state   = 3407,
)

## 2. Code-execution sandbox (subprocess-isolated)

In [ ]:
def execute_code_safely(code_str, test_cases=None, timeout_seconds=10):
    """Subprocess-isolated execution with a hard OS timeout, used by the correctness reward."""
    if test_cases is None:
        test_cases = []

    full_script = code_str + "\n\n" + "\n".join(test_cases)
    tmp_path = None
    proc = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
            f.write(full_script)
            tmp_path = f.name

        proc = subprocess.Popen(
            ["python", tmp_path],
            stdout = subprocess.PIPE,
            stderr = subprocess.PIPE,
            text   = True,
            start_new_session = True,
        )
        try:
            stdout, stderr = proc.communicate(timeout=timeout_seconds)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            except (ProcessLookupError, PermissionError):
                pass
            try:
                proc.communicate(timeout=2)
            except subprocess.TimeoutExpired:
                pass
            return False, "Execution Failed: Time Limit Exceeded", "Runtime"

        if proc.returncode == 0:
            return True, f"Execution Successful. Stdout:\n{stdout}", "None"
        return False, f"Error:\n{stderr[-500:]}", "Runtime"

    except Exception as e:
        return False, f"Execution Failed: {e}", "Runtime"

    finally:
        if tmp_path and os.path.exists(tmp_path):
            try:
                os.unlink(tmp_path)
            except OSError:
                pass

## 3. Load GRPO dataset & filter by length

In [ ]:
!unzip grpo_train_dataset_0_6.zip

In [ ]:
!mv content/drive/MyDrive/LABD2/grpo_train_dataset_0_6-2_1 /root/grpo_train_dataset_0_6-2_1

In [ ]:
from collections import Counter
from datasets import concatenate_datasets

# attempt_index: 0 = solve from scratch, >=1 = self-correction (recovery) turns.
DATASET_DIR = "/root/grpo_train_dataset_0_6-2_1"
train_dataset = load_from_disk(DATASET_DIR)

def calc_len(ex):
    toks = tokenizer(ex["prompt"], truncation=False, add_special_tokens=False)
    return {"prompt_length": len(toks["input_ids"])}

with_len = train_dataset.map(calc_len)

SCRATCH_PER_RECOVERY = 0.75   # 1.0 -> 50% recovery, 2.0 -> 33%, 0 -> recovery-only
RECOVERY_DUP         = 1
SEED                 = 3407

pool     = with_len.filter(lambda x: x["prompt_length"] <= MAX_PROMPT_LEN)
recovery = pool.filter(lambda x: x["attempt_index"] >= 1)
scratch  = pool.filter(lambda x: x["attempt_index"] == 0)

n_scratch = min(len(scratch), int(len(recovery) * SCRATCH_PER_RECOVERY))
scratch_sampled = scratch.shuffle(seed=SEED).select(range(n_scratch))

filtered = concatenate_datasets([recovery] * RECOVERY_DUP + [scratch_sampled])
filtered = filtered.shuffle(seed=SEED).remove_columns(["prompt_length"])

In [ ]:
EXECUTE_RE = re.compile(r"<execute>(.*?)</execute>", re.DOTALL)


def extract_execute_block(text):
    m = EXECUTE_RE.search(text)
    return m.group(1).strip() if m else None


_REWARD_POOL = concurrent.futures.ThreadPoolExecutor(max_workers=4)


def _score_correctness_one(completion, tests):
    code = extract_execute_block(completion)
    if not code:
        return 0.0
    try:
        success, _, _ = execute_code_safely(code, tests, timeout_seconds=10)
        return 3.0 if success else 0.0
    except Exception:
        return 0.0


# Reward 1: correctness -- +3 if the <execute> block passes all tests, else 0.
def correctness_reward_func(prompts, completions, answer, **kwargs):
    futures = [
        _REWARD_POOL.submit(_score_correctness_one, c, t)
        for c, t in zip(completions, answer)
    ]
    return [f.result() for f in futures]


# Reward 2: stuck-think -- -1 if <think> was opened but never closed.
def stuck_think_reward_func(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        if "<think>" in completion and "</think>" not in completion:
            rewards.append(-1.0)
        else:
            rewards.append(0.0)
    return rewards


# Reward 3: format -- -2 for a hallucinated <tool_response>, -1 for no <execute> when not stuck-think.
def format_reward_func(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        score = 0.0
        if "<tool_response>" in completion:
            score -= 2.0
        has_execute = "<execute>" in completion and "</execute>" in completion
        is_stuck    = "<think>" in completion and "</think>" not in completion
        if (not has_execute) and (not is_stuck):
            score -= 1.0
        rewards.append(score)
    return rewards

## 4. GRPO training

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from vllm import SamplingParams

vllm_sampling_params = SamplingParams(
    temperature        = 0.7,
    top_p              = 0.95,
    top_k              = 20,
    min_p              = 0.0,
    repetition_penalty = 1.05,
    seed               = 3407,
    stop               = [tokenizer.eos_token, "<|im_end|>"],
    include_stop_str_in_output = True,
)

training_args = GRPOConfig(
    vllm_sampling_params       = vllm_sampling_params,
    temperature                = 0.7,
    learning_rate              = 5e-6,
    weight_decay               = 0.01,
    warmup_ratio               = 0.1,
    lr_scheduler_type          = "linear",
    optim                      = "adamw_8bit",
    logging_steps              = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 8,
    num_generations            = 8,
    max_prompt_length          = MAX_PROMPT_LEN,
    max_completion_length      = MAX_COMPLETION_LEN,
    num_train_epochs           = 1,
    save_steps                 = 10,
    max_grad_norm              = 1.0,
    dataloader_num_workers     = 0,
    report_to                  = "none",
    output_dir                 = "/mnt/train/grpo_outputs_0.6B_2.1",
)

trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [
        correctness_reward_func,
        stuck_think_reward_func,
        format_reward_func,
    ],
    args          = training_args,
    train_dataset = filtered,
)

In [ ]:
trainer.train()

## 5. Save & push to Hub

In [ ]:
# LoRA adapters
model.push_to_hub(
    "moazeldegwy/Qwen3-0.6B-LABD-2.1-GRPO-Adapters",
    token = hf_token,
)
tokenizer.push_to_hub(
    "moazeldegwy/Qwen3-0.6B-LABD-2.1-GRPO-Adapters",
    token = hf_token,
)

In [ ]:
# Merged 16-bit model
model.push_to_hub_merged(
    "moazeldegwy/Qwen3-0.6B-LABD-2.1-GRPO",
    tokenizer,
    save_method = "merged_16bit",
    token       = hf_token,
)